In [ ]:
!pip install -U pandas numpy scikit-learn matplotlib seaborn tqdm requests

In [ ]:
import os
import re
import time
import json
import unicodedata

import numpy as np
import pandas as pd
import requests

import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)

print("Everything imported successfully.")

In [ ]:
# ============================================================
# LM STUDIO CONFIGURATION
# ============================================================

LM_STUDIO_URL = "http://localhost:1234/v1"

# IMPORTANT:
# We will automatically detect the model
# from LM Studio.

MODEL_NAME = None

DATA_PATH = "Dataset.csv"

OUT_DIR = "./lmstudio_results"

SEED = 42

DEV_SIZE = 500

SHOTS_LIST = [0, 3]

TEMPERATURE = 0.0

MAX_NEW_TOKENS = 4

CLASS_NAMES = {
    0: "NoMHD",
    1: "MHD"
}

os.makedirs(OUT_DIR, exist_ok=True)

np.random.seed(SEED)

sns.set_theme(style="whitegrid")

print("LM Studio URL:", LM_STUDIO_URL)

In [ ]:
# ============================================================
# GET MODELS FROM LM STUDIO
# ============================================================

response = requests.get(
    f"{LM_STUDIO_URL}/models",
    timeout=10
)

response.raise_for_status()

models_data = response.json()

print("Models available in LM Studio:\n")

for model in models_data.get("data", []):
    print(
        "ID:",
        model.get("id")
    )

In [ ]:
MODEL_NAME = models_data["data"][0]["id"]

print("Using model:")
print(MODEL_NAME)

In [ ]:
# ============================================================
# SIMPLE LM STUDIO TEST
# ============================================================

messages = [
    {
        "role": "user",
        "content": "বাংলায় বলো, ১ + ১ কত?"
    }
]

payload = {
    "model": MODEL_NAME,
    "messages": messages,
    "temperature": 0.0,
    "max_tokens": 20
}

response = requests.post(
    f"{LM_STUDIO_URL}/chat/completions",
    json=payload,
    timeout=120
)

response.raise_for_status()

result = response.json()

print(
    result["choices"][0]["message"]["content"]
)

In [ ]:
# ============================================================
# LOAD DATASET
# ============================================================

def clean_text(s):
    return " ".join(
        unicodedata.normalize(
            "NFC",
            str(s)
        ).split()
    )


df = pd.read_csv(
    DATA_PATH,
    header=None,
    names=["text", "label"]
)

print("Original:", df.shape)

df = df.dropna(
    subset=["text", "label"]
).copy()

df["text"] = df["text"].apply(
    clean_text
)

df["label"] = pd.to_numeric(
    df["label"],
    errors="coerce"
)

df = df.dropna(
    subset=["label"]
)

df["label"] = df["label"].astype(int)

df = df[
    df["label"].isin([0, 1])
]

df = df[
    df["text"].str.len() > 0
]

df = (
    df
    .drop_duplicates(subset="text")
    .reset_index(drop=True)
)

print("\nUsable samples:", len(df))

print("\nClass distribution:")
print(
    df["label"]
    .map(CLASS_NAMES)
    .value_counts()
)

In [ ]:
# ============================================================
# DEVELOPMENT / TEST SPLIT
# ============================================================

dev_df, test_df = train_test_split(
    df,
    train_size=DEV_SIZE,
    stratify=df["label"],
    random_state=SEED
)

dev_df = dev_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(
    "Development:",
    len(dev_df)
)

print(
    "Test:",
    len(test_df)
)

print("\nDevelopment:")
print(
    dev_df["label"]
    .map(CLASS_NAMES)
    .value_counts()
)

print("\nTest:")
print(
    test_df["label"]
    .map(CLASS_NAMES)
    .value_counts()
)

In [ ]:
# ============================================================
# SELECT 3-SHOT EXAMPLES
# ============================================================

rng = np.random.RandomState(SEED)

demo_0 = (
    dev_df[
        dev_df["label"] == 0
    ]
    .sample(
        1,
        random_state=SEED
    )
)

demo_1 = (
    dev_df[
        dev_df["label"] == 1
    ]
    .sample(
        1,
        random_state=SEED + 1
    )
)

third_label = rng.choice([0, 1])

demo_2 = (
    dev_df[
        dev_df["label"] == third_label
    ]
    .sample(
        1,
        random_state=SEED + 2
    )
)

demo_df = pd.concat(
    [
        demo_0,
        demo_1,
        demo_2
    ],
    ignore_index=True
)

demo_df = demo_df.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

for i, row in demo_df.iterrows():

    print("=" * 60)

    print(
        f"Example {i+1}"
    )

    print(
        "Label:",
        row["label"],
        CLASS_NAMES[row["label"]]
    )

    print(
        "Text:",
        row["text"][:500]
    )

In [ ]:
SYSTEM_PROMPT = """
তুমি একটি বাংলা মানসিক-স্বাস্থ্য টেক্সট classification system।

তোমার কাজ হলো প্রদত্ত বাংলা লেখাটিকে দুটি লেবেলের একটিতে শ্রেণিবদ্ধ করা।

লেবেল 1 (MHD):
লেখক নিজের বা নিজের খুব কাছের কারও মানসিক বা আবেগীয় কষ্ট,
উপসর্গ, আচরণগত সমস্যা, মানসিক চাপ, সাহায্যের প্রয়োজন,
অথবা মানসিক স্বাস্থ্য সম্পর্কিত কোনো সমস্যা প্রকাশ করছেন।

লেবেল 0 (NoMHD):
লেখক নিজের মানসিক স্বাস্থ্য সমস্যা বা সাহায্যের প্রয়োজন
প্রকাশ করছেন না।

শুধু কোনো নির্দিষ্ট শব্দ যেমন মৃত্যু, কষ্ট, কান্না,
সুইসাইড বা ডিপ্রেশন আছে কি না তার ভিত্তিতে সিদ্ধান্ত নেবে না।
লেখাটির সামগ্রিক অর্থ বিবেচনা করবে।

শুধুমাত্র একটি সংখ্যা লিখবে:

1

অথবা

0

অন্য কোনো শব্দ বা ব্যাখ্যা লিখবে না।
"""


USER_TEMPLATE = """
লেখা:
\"\"\"{text}\"\"\"

প্রশ্ন:
লেখক কি নিজের মানসিক স্বাস্থ্য সংক্রান্ত কষ্ট,
সমস্যা বা সাহায্যের প্রয়োজন প্রকাশ করছেন?

শুধুমাত্র 1 অথবা 0 লিখুন।

উত্তর:
"""

In [ ]:
def build_messages(
    text,
    shots
):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        }
    ]

    # 3-shot demonstrations
    for shot_text, shot_label in shots:

        messages.append(
            {
                "role": "user",
                "content": USER_TEMPLATE.format(
                    text=shot_text
                )
            }
        )

        messages.append(
            {
                "role": "assistant",
                "content": str(
                    shot_label
                )
            }
        )

    # Actual test text
    messages.append(
        {
            "role": "user",
            "content": USER_TEMPLATE.format(
                text=text
            )
        }
    )

    return messages

In [ ]:
def parse_prediction(content):

    if content is None:
        return None, False

    content = str(
        content
    ).strip()

    if content == "0":
        return 0, True

    if content == "1":
        return 1, True

    return None, False

In [ ]:
def score_one(
    text,
    shots
):

    messages = build_messages(
        text,
        shots
    )

    payload = {
        "model": MODEL_NAME,

        "messages": messages,

        "temperature": TEMPERATURE,

        "max_tokens": MAX_NEW_TOKENS,

        "stream": False
    }

    for attempt in range(4):

        try:

            response = requests.post(
                f"{LM_STUDIO_URL}/chat/completions",
                json=payload,
                timeout=300
            )

            response.raise_for_status()

            data = response.json()

            content = (
                data["choices"][0]["message"]
                .get("content", "")
            )

            prediction, valid = (
                parse_prediction(
                    content
                )
            )

            usage = data.get(
                "usage",
                {}
            )

            return {
                "prediction": prediction,
                "valid": valid,
                "raw_output": content,
                "prompt_tokens": usage.get(
                    "prompt_tokens",
                    0
                ),
                "completion_tokens": usage.get(
                    "completion_tokens",
                    0
                ),
                "total_tokens": usage.get(
                    "total_tokens",
                    0
                ),
                "error": ""
            }

        except Exception as e:

            if attempt == 3:

                return {
                    "prediction": None,
                    "valid": False,
                    "raw_output": "",
                    "prompt_tokens": 0,
                    "completion_tokens": 0,
                    "total_tokens": 0,
                    "error": str(e)
                }

            time.sleep(
                2 ** attempt
            )

In [ ]:
# ============================================================
# TEST ONE DATASET ROW
# ============================================================

sample_text = test_df.iloc[0]["text"]

result = score_one(
    sample_text,
    []
)

print("TEXT:")
print(sample_text)

print("\nMODEL OUTPUT:")
print(result["raw_output"])

print("\nPREDICTION:")
print(result["prediction"])

print("\nVALID:")
print(result["valid"])

print("\nTOKENS:")
print(result["total_tokens"])

print("\nERROR:")
print(result["error"])

In [ ]:
# ============================================================
# PILOT DATASET
# ============================================================

PILOT_SIZE = 20

pilot_df = (
    test_df
    .sample(
        PILOT_SIZE,
        random_state=SEED
    )
    .reset_index(drop=True)
)

print(
    "Pilot samples:",
    len(pilot_df)
)

In [ ]:
# ============================================================
# RUN EVALUATION
# ============================================================

def run_evaluation(
    evaluation_df
):

    results = {}

    texts = evaluation_df[
        "text"
    ].tolist()

    labels = evaluation_df[
        "label"
    ].astype(int).tolist()

    # -------------------------------
    # Create 0-shot and 3-shot
    # -------------------------------

    shot_configs = {

        "Zero-shot": [],

        "3-shot": [
            (
                row["text"],
                int(row["label"])
            )
            for _, row
            in demo_df.iterrows()
        ]
    }

    for setting_name, shots in shot_configs.items():

        print("\n")
        print("=" * 70)
        print(setting_name)
        print("=" * 70)

        predictions = []
        valid_flags = []
        raw_outputs = []

        prompt_tokens = []
        completion_tokens = []
        total_tokens = []

        errors = []

        pbar = tqdm(
            range(len(texts)),
            desc=setting_name
        )

        for i in pbar:

            result = score_one(
                texts[i],
                shots
            )

            predictions.append(
                result["prediction"]
            )

            valid_flags.append(
                result["valid"]
            )

            raw_outputs.append(
                result["raw_output"]
            )

            prompt_tokens.append(
                result["prompt_tokens"]
            )

            completion_tokens.append(
                result["completion_tokens"]
            )

            total_tokens.append(
                result["total_tokens"]
            )

            errors.append(
                result["error"]
            )

            valid_mask = [
                x is not None
                for x in predictions
            ]

            if any(valid_mask):

                true_temp = np.array(labels)[
                    valid_mask
                ]

                pred_temp = np.array([
                    predictions[j]
                    for j in range(
                        len(predictions)
                    )
                    if valid_mask[j]
                ])

                acc = accuracy_score(
                    true_temp,
                    pred_temp
                )

                pbar.set_postfix(
                    accuracy=f"{acc:.3f}",
                    valid=f"{np.mean(valid_flags):.3f}"
                )

        result_df = evaluation_df[
            ["text", "label"]
        ].copy()

        result_df["prediction"] = predictions

        result_df["valid_format"] = (
            valid_flags
        )

        result_df["raw_output"] = (
            raw_outputs
        )

        result_df["prompt_tokens"] = (
            prompt_tokens
        )

        result_df["completion_tokens"] = (
            completion_tokens
        )

        result_df["total_tokens"] = (
            total_tokens
        )

        result_df["error"] = errors

        results[
            setting_name
        ] = result_df

        filename = (
            setting_name
            .lower()
            .replace("-", "_")
        )

        result_df.to_csv(
            f"{OUT_DIR}/{filename}.csv",
            index=False,
            encoding="utf-8-sig"
        )

    return results

In [ ]:
pilot_results = run_evaluation(
    pilot_df
)

In [ ]:
# ============================================================
# PILOT METRICS
# ============================================================

for setting_name, result_df in pilot_results.items():

    valid_df = result_df[
        result_df["valid_format"]
    ]

    print("\n" + "=" * 70)
    print(setting_name)
    print("=" * 70)

    print(
        "Total:",
        len(result_df)
    )

    print(
        "Valid:",
        len(valid_df)
    )

    if len(valid_df) > 0:

        print(
            "Accuracy:",
            accuracy_score(
                valid_df["label"],
                valid_df["prediction"]
            )
        )

        print(
            classification_report(
                valid_df["label"],
                valid_df["prediction"],
                target_names=[
                    "NoMHD",
                    "MHD"
                ],
                digits=4,
                zero_division=0
            )
        )

In [ ]:
FULL_EVALUATION = True

In [ ]:
if FULL_EVALUATION:

    full_results = run_evaluation(
        test_df
    )

    print(
        "Full evaluation completed."
    )

In [ ]:
# ============================================================
# FULL METRICS
# ============================================================

metric_rows = []

for setting_name, result_df in full_results.items():

    valid_df = result_df[
        result_df["valid_format"]
    ].copy()

    if len(valid_df) == 0:
        continue

    y_true = valid_df[
        "label"
    ].astype(int)

    y_pred = valid_df[
        "prediction"
    ].astype(int)

    precision, recall, f1, support = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=[0, 1],
            zero_division=0
        )
    )

    macro_p, macro_r, macro_f1, _ = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        )
    )

    metric_rows.append({

        "Setting": setting_name,

        "N": len(result_df),

        "Valid N": len(valid_df),

        "Format Adherence":
            len(valid_df) / len(result_df),

        "Accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "Precision NoMHD":
            precision[0],

        "Recall NoMHD":
            recall[0],

        "F1 NoMHD":
            f1[0],

        "Precision MHD":
            precision[1],

        "Recall MHD":
            recall[1],

        "F1 MHD":
            f1[1],

        "Macro Precision":
            macro_p,

        "Macro Recall":
            macro_r,

        "Macro F1":
            macro_f1,

        "Total Tokens":
            result_df[
                "total_tokens"
            ].sum()
    })


metrics_df = pd.DataFrame(
    metric_rows
)

print(
    metrics_df.round(4)
)

metrics_df.to_csv(
    f"{OUT_DIR}/metrics_summary.csv",
    index=False
)

In [ ]:
# ============================================================
# CONFUSION MATRICES
# ============================================================

for setting_name, result_df in full_results.items():

    valid_df = result_df[
        result_df["valid_format"]
    ].copy()

    if len(valid_df) == 0:
        continue

    cm = confusion_matrix(
        valid_df["label"],
        valid_df["prediction"],
        labels=[0, 1]
    )

    plt.figure(
        figsize=(5, 4)
    )

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        xticklabels=[
            "NoMHD",
            "MHD"
        ],
        yticklabels=[
            "NoMHD",
            "MHD"
        ]
    )

    plt.title(
        f"{MODEL_NAME}\n{setting_name}"
    )

    plt.xlabel(
        "Predicted"
    )

    plt.ylabel(
        "True"
    )

    plt.tight_layout()

    filename = (
        f"CM_{setting_name}"
        .replace("-", "_")
    )

    plt.savefig(
        f"{OUT_DIR}/{filename}.png",
        dpi=200,
        bbox_inches="tight"
    )

    plt.show()